## Project El Nino effects 

Import packages

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import os
from dask_gateway import Gateway
import hvplot.xarray

warnings.simplefilter('ignore') # filter some warning messages
xr.set_options(display_style="html")  #display dataset nicely 

## Set up Dask Cluster

In [ ]:
# 1. Initialize Gateway
gateway = Gateway()

# 2. Configure Cluster Options
options = gateway.cluster_options()
# Ensures workers use the same environment as your notebook
options.image = os.environ.get('JUPYTER_IMAGE', 'default') 

# 3. Create and Scale Cluster
cluster = gateway.new_cluster(options)

# We set a minimum of 4 workers to start immediately, 
# but allow scaling up to 30 for the heavy climatology math.
cluster.adapt(minimum=4, maximum=40)

# 4. Connect the Client
client = cluster.get_client()

# Display the dashboard link
print(f"Cluster is up! View progress here: {cluster.dashboard_link}")
client

In [ ]:
if cluster_type == 'Coiled':
    import coiled
    cluster = coiled.Cluster(
        region="us-west-2",
        arm=True,   # run on ARM to save energy & cost
        worker_vm_types=["t4g.small"],  # cheap, small ARM instances, 2cpus, 2GB RAM
        worker_options={'nthreads':2},
        n_workers=30,
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        name='coawst',   # Dask cluster name
        software='protocoast-develop-arm',  # Conda environment name
        workspace='osc-aws',
        timeout=180   # leave cluster running for 3 min in case we want to use it again
    )

    client = cluster.get_client()

#Explore data

In [ ]:
%%time

ds_sst = xr.open_zarr('https://mur-sst.s3.us-west-2.amazonaws.com/zarr-v1',consolidated=True)

ds_sst

#Get only SST 

In [ ]:
%%time

sst = ds_sst['analysed_sst']

#Spatial slice of data: 20°N - 20°S, 100°E - 160°W

In [ ]:
sst_360 = sst.assign_coords(lon=(sst.lon % 360)).sortby('lon')
sst_sliced = sst_360.sel(
    lat=slice(-20, 20), 
    lon=slice(100, 200)
)

In [ ]:
#do not run
sst_sliced = sst.sel(
    lat=slice(-20, 20), 
    lon=slice(100, 180) # This gets 100E to the Date Line
).combine_first(
    sst.sel(lat=slice(-20, 20), lon=slice(-100, -160)) # This gets Date Line to 60W
)
sst_sliced

#Monthly mean, climatology, anomaly

In [ ]:
#Conversion Kelvin to Celsius
sst_celsius = sst_sliced - 273.15

In [ ]:
#Create monthly mean 
sst_monthly = sst_celsius.resample(time='1MS').mean('time', keep_attrs=True, skipna=True)
sst_monthly

In [ ]:
#Create monthly climatology
climatology_mean_monthly = sst_monthly.groupby('time.month').mean('time', keep_attrs=True, skipna=True)
climatology_mean_monthly

In [ ]:
#Create monthly anomaly
sst_anomaly_monthly = sst_monthly.groupby('time.month') - climatology_mean_monthly

In [ ]:
# Isolate 2002 
sst_2002 = sst_monthly.sel(time='2002')

In [ ]:
%%time 
#Coarsening data by 10 

sst_2002_small = sst_2002.coarsen(lat=10, lon=10, boundary='trim').mean()
da = sst_2002_small.load()

In [ ]:
# Hvplot 
import holoviews as hv

sst_2002 = sst_2002.hvplot.quadmesh( 
    x='lon', 
    y='lat', 
    cmap='viridis', 
    title="SST Monthly Mean: 2002",
    groupby='time',
    geo=True,
    rasterize=True # Use this if the data is large to keep it fast
)


sst_2002_light.hvplot.quadmesh(
    x='lon', 
    y='lat', 
    rasterize=True, 
    geo=True, 
    cmap='viridis',
    title="Monthly Mean SST 2002 (10km Coarsened)",
    groupby='time', # Adds the month slider
    width=700,
    height=400,
    clim=(20, 32)   # Fixes the color scale so it doesn't jump as you slide
)



In [ ]:
sst_2002.plot(cmap="RdBu_r")

### Global Ocean Chlorophyll-a trend map from Observations Reprocessing- Copernicus

In [ ]:
import os
import fsspec
import xarray as xr
import hvplot.xarray
import intake
import cf_xarray
import numpy as np
import panel as pn
from matplotlib import path
import xoak
import zarr
import copernicusmarine

### Open dataset

In [ ]:
%%time 
ds1 = copernicusmarine.open_dataset(dataset_id='c3s_obs-oc_glo_bgc-plankton_my_l4-multi-4km_P1M')

In [ ]:
ds1

In [ ]:
#do not open
%%time
ds2 = copernicusmarine.open_dataset(dataset_id='omi_health_chl_global_oceancolour_trend')

In [ ]:
import hvplot.xarray
import panel as pn
pn.extension()

### El Nino conditions (winter 2015 - 2016)

In [ ]:
#January 2016 (El Nino year) 
da = ds1['CHL'].sel(time='2016-01-01 00:00', method='nearest').load()

In [ ]:
da
da.hvplot(x='longitude', y='latitude', rasterize=True, geo=True, cmap='turbo', tiles='OSM')

In [ ]:
da.hvplot(x='longitude', 
          y='latitude', 
          rasterize=True, 
          geo=True, 
          cmap='YlGn',       # 'YlGn' (Yellow-Green) is standard for plants/Chl
          clim=(0, 15),      # Focus the color range between 0 and 15
          tiles='OSM',
          title="Chlorophyll Concentration (mg/m3)")

### Neutral conditions (winter 2017 - 2018)

In [ ]:
#January 2018 (NOT El Nino year) 
da2 = ds1['CHL'].sel(time='2018-01-01 00:00', method='nearest').load()

### Chlorophyll concentration in my area of interest: Peruvian coast (25° S – 5° S, 90° W – 65° W)

In [ ]:
#Define my area
ch_small = ds1['CHL'].sel(longitude=slice(-90, -65), latitude=slice(-25, -5))

In [ ]:
ch_small

In [ ]:
#Chlorophyll data for my area (1997-2024) => set it for January 1st 2026
ch_small.hvplot(x='longitude', 
          y='latitude', 
          rasterize=True, 
          geo=True, 
          cmap='YlGn',       # 'YlGn' (Yellow-Green) is standard for plants/Chl
          clim=(0, 15),      # Focus the color range between 0 and 15
          tiles='OSM',
          title="Chlorophyll Concentration (mg/m3)")

In [ ]:
da2.hvplot(x='longitude', 
          y='latitude', 
          rasterize=True, 
          geo=True, 
          cmap='YlGn',       # 'YlGn' (Yellow-Green) is standard for plants/Chl
          clim=(0, 15),      # Focus the color range between 0 and 15
          tiles='OSM',
          title="Chlorophyll Concentration (mg/m3)")

### Time series spectrogram of Chlorophyll concentration on the Peruvian coast

In [ ]:
%%time
# Monthly Chl concentration for the time and point of interest
monthly_Ch = ds1['CHL'].sel(time=slice('2014', '2020')) \
                       .sel(longitude=80, latitude=10, method='nearest') \
                       .load()

In [ ]:
monthly_Ch

In [ ]:
# Plot 
monthly_Ch.hvplot(grid=True, ylabel= "Chlorophyll concentration (mg/m3)", title="Monthly Chlorophyll concentration (mg/m3), 2014-2020")

In [ ]:
cluster.shutdown()